In [1]:
# ============================================================
#  1 - Install dari local wheels
# ============================================================
import subprocess, sys, os, glob
 
WHEELS_DIR = "/kaggle/input/datasets/herdinthorikn/capstonedataset/paddle_wheels"
 
# Urutan penting: paddlepaddle dulu, baru yang lain
wheels_order = [
    "paddlepaddle_gpu*.whl",
    "shapely*.whl",
    "pyclipper*.whl",
    "onnx-*.whl",
    "onnxruntime_gpu*.whl",
    "paddle2onnx*.whl",
    # "GPUtil-*.tar.gz",
    # "paddleocr*.whl",
]
 
for pattern in wheels_order:
    matches = glob.glob(os.path.join(WHEELS_DIR, pattern))
    if matches:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            matches[0],
            "--no-index",
            "--find-links", WHEELS_DIR,
            "--quiet"
        ])
        print(f"✓ {os.path.basename(matches[0])}")
    else:
        print(f"✗ NOT FOUND: {pattern}")
        
paddleocr_whl = glob.glob(os.path.join(WHEELS_DIR, "paddleocr*.whl"))
if paddleocr_whl:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        paddleocr_whl[0],
        "--no-deps",    
        "--quiet"
    ])
    print(f"✓ {os.path.basename(paddleocr_whl[0])} (no-deps)")
else:
    print("✗ NOT FOUND: paddleocr*.whl")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.


✓ paddlepaddle_gpu-3.1.0-cp312-cp312-linux_x86_64.whl
✓ shapely-2.1.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
✓ pyclipper-1.4.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
✓ onnx-1.17.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
✓ onnxruntime_gpu-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
✓ paddle2onnx-2.1.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
✓ paddleocr-3.1.1-py3-none-any.whl (no-deps)


In [2]:
# Install lmdb dari wheels kalau ada, fallback ke pip
import glob, subprocess, sys

lmdb_whl = glob.glob("/kaggle/input/datasets/herdinthorikn/capstonedataset/paddle_wheels/lmdb*.whl")
if lmdb_whl:
    subprocess.check_call([sys.executable, "-m", "pip", "install", lmdb_whl[0], "--quiet"])
else:
    # lmdb tidak ada di wheels — perlu internet ON untuk ini
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lmdb", "--quiet"])
print("✓ lmdb installed")

subprocess.check_call([sys.executable, "-m", "pip", "install", "rapidfuzz", "--quiet"])
print("✓ rapidfuzz installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 15.0 MB/s eta 0:00:00
✓ lmdb installed
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.6 MB/s eta 0:00:00
✓ rapidfuzz installed


In [3]:
# ============================================================
#  2 — Unzip repo + data + buat direktori
# ============================================================
import os, shutil
from pathlib import Path
 
# PaddleOCR repo
repo_src = "/kaggle/input/datasets/herdinthorikn/capstonedataset/paddleocr_repo"
extract_dir = "/kaggle/working/PaddleOCR"

if not Path(extract_dir).exists():
    shutil.copytree(repo_src, extract_dir)
    print("✓ PaddleOCR repo copied")
else:
    print("✓ PaddleOCR repo already exists")

assert Path(f"{extract_dir}/tools/train.py").exists(), \
    "ERROR: tools/train.py tidak ditemukan"
 
# paddle_data
data_src = "/kaggle/input/datasets/herdinthorikn/capstonedataset/paddle_data"
if not Path("/kaggle/working/paddle_data").exists():
    shutil.copytree(data_src, "/kaggle/working/paddle_data")
    print("✓ paddle_data copied")
else:
    print("✓ paddle_data already exists")
 
# Verifikasi file label ada
for f in ["det_train.txt", "det_val.txt", "rec_train.txt", "rec_val.txt"]:
    path = f"/kaggle/working/paddle_data/{f}"
    count = sum(1 for _ in open(path)) if Path(path).exists() else 0
    status = "✓" if count > 0 else "✗ MISSING"
    print(f"{status} {f}: {count} baris")
 
# Buat semua direktori output
for d in [
    "/kaggle/working/logs",
    "/kaggle/working/output/det_finetune",
    "/kaggle/working/output/rec_finetune",
    "/kaggle/working/output/det_infer",
    "/kaggle/working/output/rec_infer",
    "/kaggle/working/onnx_models",
    "/kaggle/working/configs",
]:
    os.makedirs(d, exist_ok=True)
 
print("✓ Semua direktori siap")

✓ PaddleOCR repo copied
✓ paddle_data copied
✓ det_train.txt: 17 baris
✓ det_val.txt: 3 baris
✓ rec_train.txt: 20319 baris
✓ rec_val.txt: 3587 baris
✓ Semua direktori siap


In [4]:
# ============================================================
#  3 — Verifikasi GPU + PaddlePaddle
# ============================================================
import paddle
print(f"PaddlePaddle version : {paddle.__version__}")
print(f"CUDA compiled        : {paddle.is_compiled_with_cuda()}")
print(f"GPU count            : {paddle.device.cuda.device_count()}")
paddle.utils.run_check()
# Jangan lanjut ke Cell 4 kalau cell ini error

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


PaddlePaddle version : 3.1.0
CUDA compiled        : True
GPU count            : 2
Running verify PaddlePaddle program ... 


I0604 00:42:57.729761    25 pir_interpreter.cc:1524] New Executor is Running ...
W0604 00:42:57.732724    25 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 11.8


PaddlePaddle works well on 1 GPU.


I0604 00:42:57.955240    25 pir_interpreter.cc:1547] pir interpreter is running by multi-thread mode ...
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
======================= Modified FLAGS detected =======================
FLAGS(name='FLAGS_cudnn_dir', current_value='/usr/local/lib/python3.12/dist-packages/paddle/../nvidia/cudnn/lib', default_value='')
FLAGS(name='FLAGS_curand_dir', current_value=

PaddlePaddle works well on 2 GPUs.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


In [5]:
import json

def fix_det_label_file(input_path, output_path):
    with open(input_path) as fin, open(output_path, "w") as fout:
        for line in fin:
            img_path, label_str = line.strip().split("\t", 1)
            annotations = json.loads(label_str)
            fixed = []
            for ann in annotations:
                points = ann["points"]
                # Pastikan semua koordinat integer
                fixed_points = [[int(p[0]), int(p[1])] for p in points]
                fixed.append({
                    "transcription": ann["transcription"],
                    "points": fixed_points
                })
            fout.write(img_path + "\t" + json.dumps(fixed) + "\n")
    print(f"✓ Fixed: {output_path}")

fix_det_label_file(
    "/kaggle/working/paddle_data/det_train.txt",
    "/kaggle/working/paddle_data/det_train_fixed.txt"
)
fix_det_label_file(
    "/kaggle/working/paddle_data/det_val.txt", 
    "/kaggle/working/paddle_data/det_val_fixed.txt"
)

✓ Fixed: /kaggle/working/paddle_data/det_train_fixed.txt
✓ Fixed: /kaggle/working/paddle_data/det_val_fixed.txt


In [6]:
def fix_rec_label_file(input_path, output_path):
    with open(input_path, encoding="utf-8", errors="ignore") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:
        for line in fin:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t", 1)
            if len(parts) != 2:
                continue  # skip baris malformed
            img_path, transcription = parts
            if not transcription:
                continue  # skip label kosong
            fout.write(img_path + "\t" + transcription + "\n")
    print(f"✓ Fixed: {output_path}")

fix_rec_label_file(
    "/kaggle/working/paddle_data/rec_train.txt",
    "/kaggle/working/paddle_data/rec_train_fixed.txt"
)
fix_rec_label_file(
    "/kaggle/working/paddle_data/rec_val.txt", 
    "/kaggle/working/paddle_data/rec_val_fixed.txt"
)

✓ Fixed: /kaggle/working/paddle_data/rec_train_fixed.txt
✓ Fixed: /kaggle/working/paddle_data/rec_val_fixed.txt


In [7]:
# ============================================================
#  4 — Buat char dict dari data rec aktual
# ============================================================
char_set = set()
with open("/kaggle/working/paddle_data/rec_train_fixed.txt", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            char_set.update(parts[1])
 
char_list = sorted(char_set)
dict_path = "/kaggle/working/configs/rec_char_dict.txt"
with open(dict_path, "w", encoding="utf-8") as f:
    for c in char_list:
        f.write(c + "\n")
 
print(f"✓ Char dict: {len(char_list)} karakter unik")
print(f"  Sample: {char_list[:30]}")

✓ Char dict: 88 karakter unik
  Sample: [' ', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>']


In [8]:
# ============================================================
#  PATCHING PADDLEOCR UNTUK NUMPY 2.0+ (Surgical Fix)
# ============================================================
import os

file_path = "/kaggle/working/PaddleOCR/ppocr/data/imaug/random_crop_data.py"

if os.path.exists(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Mencari baris kode legacy dan menggantinya dengan sintaks yang kompatibel dengan NumPy 2.0
    target_legacy = "xx = int(np.random.choice(axis, size=1))"
    target_modern = "xx = int(np.random.choice(axis, size=1)[0])"
    
    if target_legacy in content:
        content = content.replace(target_legacy, target_modern)
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f"✓ Patch berhasil diterapkan pada: {file_path}")
    elif target_modern in content:
        print("✓ File sudah di-patch sebelumnya.")
    else:
        print("⚠ Peringatan: Baris target tidak ditemukan. Cek versi file.")
else:
    print(f"✗ File tidak ditemukan: {file_path}")

✓ Patch berhasil diterapkan pada: /kaggle/working/PaddleOCR/ppocr/data/imaug/random_crop_data.py


In [9]:
# ============================================================
#  6.5 — AUGMENTATION UNTUK DETECTION 
# ============================================================
import cv2
import os
import albumentations as A
from tqdm import tqdm

input_label_path = "/kaggle/working/paddle_data/det_train_fixed.txt"
output_label_path = "/kaggle/working/paddle_data/det_train_augmented.txt"
aug_img_dir = "/kaggle/working/paddle_data/aug_images"

os.makedirs(aug_img_dir, exist_ok=True)

# Pipeline augmentasi HANYA pada level piksel
transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
    A.ImageCompression(quality_lower=60, quality_upper=95, p=0.4)
])

NUM_AUGMENTATIONS = 50 

total_images = 0
with open(input_label_path, "r") as fin, open(output_label_path, "w") as fout:
    lines = fin.readlines()
    for line in tqdm(lines, desc="Menggandakan Data Deteksi"):
        line = line.strip()
        if not line: continue
        
        img_path, label_str = line.split("\t", 1)
        
        # 1. Tulis gambar asli ke file label baru
        fout.write(f"{img_path}\t{label_str}\n")
        total_images += 1
        
        # Load gambar asli dari disk
        image = cv2.imread(img_path)
        if image is None:
            continue
            
        img_basename = os.path.basename(img_path).split('.')[0]
        
        # 2. Cetak versi augmentasi
        for i in range(NUM_AUGMENTATIONS):
            augmented = transform(image=image)
            aug_img = augmented["image"]
            
            # Simpan piksel gambar baru
            new_img_name = f"{img_basename}_aug_{i}.jpg"
            new_img_path = os.path.join(aug_img_dir, new_img_name)
            cv2.imwrite(new_img_path, aug_img)
            
            # Tulis baris baru menggunakan path gambar palsu namun dengan label koordinat asli
            fout.write(f"{new_img_path}\t{label_str}\n")
            total_images += 1

print(f"\n✓ Augmentasi Selesai!")
print(f"✓ File konfigurasi baru: {output_label_path}")

/tmp/ipykernel_25/519375889.py:18: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
/tmp/ipykernel_25/519375889.py:21: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=60, quality_upper=95, p=0.4)
Menggandakan Data Deteksi: 100%|██████████| 17/17 [00:38<00:00,  2.28s/it]


✓ Augmentasi Selesai!
✓ File konfigurasi baru: /kaggle/working/paddle_data/det_train_augmented.txt


In [10]:
import paddle
import yaml
import os
import gc

# =====================================================================
# 1. DEFINISI FUNGSI DALAM SATU SCOPE UNTUK MENCEGAH NAMERROR
# =====================================================================
def extract_student_weights(distill_path, clean_path):
    print(f"Menganalisis arsitektur weights: {distill_path}")
    
    # Load bobot ke memori
    distill_params = paddle.load(distill_path)
    clean_params = {}
    
    is_distilled = any(key.startswith('Student.') for key in distill_params.keys())
    
    if is_distilled:
        for key, value in distill_params.items():
            if key.startswith('Student.'):
                clean_params[key[len('Student.'):]] = value
        paddle.save(clean_params, clean_path)
        print(f"✓ Berhasil melucuti prefiks 'Student.'. Disimpan ke: {clean_path}")
    else:
        print("✓ Bukan model distilasi, tidak perlu diubah.")
        paddle.save(distill_params, clean_path)
        
    # PROTEKSI MEMORI: Hapus variabel besar dari RAM secara paksa
    del distill_params, clean_params
    gc.collect() 
    print("✓ Memori berhasil dibersihkan (Garbage Collected).\\n")

# =====================================================================
# 2. EKSEKUSI EKSTRAKSI WEIGHTS SECARA BERURUTAN
# =====================================================================
det_pretrained = "/kaggle/input/datasets/herdinthorikn/capstonedataset/paddleocr_pretrained/PP-OCRv5_mobile_det_pretrained.pdparams"
det_clean_pretrained = "/kaggle/working/PP-OCRv5_mobile_det_student_only.pdparams"

rec_pretrained = "/kaggle/input/datasets/herdinthorikn/capstonedataset/paddleocr_pretrained/PP-OCRv5_server_rec_pretrained.pdparams"
rec_clean_pretrained = "/kaggle/working/PP-OCRv5_server_rec_student_only.pdparams"

print("Memproses weights Detection...")
extract_student_weights(det_pretrained, det_clean_pretrained)

print("Memproses weights Recognition...")
extract_student_weights(rec_pretrained, rec_clean_pretrained)

# =====================================================================
# 3. PARSING DAN INJEKSI KONFIGURASI YAML
# =====================================================================
print("Membangun ulang file YAML secara dinamis...")

# A. Detection YAML
base_det_config = "/kaggle/working/PaddleOCR/configs/det/PP-OCRv5/PP-OCRv5_mobile_det.yml"
new_det_config = "/kaggle/working/configs/det_finetune.yml"

with open(base_det_config, 'r', encoding='utf-8') as f:
    det_config = yaml.safe_load(f)

det_config['Global']['pretrained_model'] = "/kaggle/working/PP-OCRv5_mobile_det_student_only"
det_config['Global']['save_model_dir'] = "/kaggle/working/output/det_finetune/"
det_config['Global']['epoch_num'] = 10
det_config['Global']['eval_batch_step'] = [0, 4]
det_config['Train']['dataset']['data_dir'] = "/kaggle/working/paddle_data/"
det_config['Train']['dataset']['label_file_list'] = ["/kaggle/working/paddle_data/det_train_augmented.txt"]
det_config['Train']['loader']['batch_size_per_card'] = 16
det_config['Train']['loader']['num_workers'] = 0
det_config['Eval']['dataset']['data_dir'] = "/kaggle/working/paddle_data/"
det_config['Eval']['dataset']['label_file_list'] = ["/kaggle/working/paddle_data/det_val_fixed.txt"]
det_config['Eval']['loader']['num_workers'] = 0

with open(new_det_config, 'w', encoding='utf-8') as f:
    yaml.dump(det_config, f, sort_keys=False)
print("✓ Config Detection siap.")

# B. Recognition YAML
base_rec_config = "/kaggle/working/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml"
new_rec_config = "/kaggle/working/configs/rec_finetune.yml"
dict_path = "/kaggle/working/configs/rec_char_dict.txt"

with open(base_rec_config, 'r', encoding='utf-8') as f:
    rec_config = yaml.safe_load(f)

rec_config['Global']['pretrained_model'] = "/kaggle/working/PP-OCRv5_server_rec_student_only"
rec_config['Global']['save_model_dir'] = "/kaggle/working/output/rec_finetune/"
rec_config['Global']['epoch_num'] = 30
rec_config['Global']['eval_batch_step'] = [0, 150]
rec_config['Global']['character_dict_path'] = dict_path
rec_config['Global']['use_space_char'] = True
rec_config['Train']['dataset']['data_dir'] = "/kaggle/working/paddle_data/"
rec_config['Train']['dataset']['label_file_list'] = ["/kaggle/working/paddle_data/rec_train_fixed.txt"]
rec_config['Train']['loader']['batch_size_per_card'] = 128
rec_config['Train']['loader']['num_workers'] = 0
rec_config['Eval']['dataset']['data_dir'] = "/kaggle/working/paddle_data/"
rec_config['Eval']['dataset']['label_file_list'] = ["/kaggle/working/paddle_data/rec_val_fixed.txt"]
rec_config['Eval']['loader']['num_workers'] = 0

with open(new_rec_config, 'w', encoding='utf-8') as f:
    yaml.dump(rec_config, f, sort_keys=False)
print("✓ Config Recognition siap.")

Memproses weights Detection...
Menganalisis arsitektur weights: /kaggle/input/datasets/herdinthorikn/capstonedataset/paddleocr_pretrained/PP-OCRv5_mobile_det_pretrained.pdparams
✓ Bukan model distilasi, tidak perlu diubah.
✓ Memori berhasil dibersihkan (Garbage Collected).\n
Memproses weights Recognition...
Menganalisis arsitektur weights: /kaggle/input/datasets/herdinthorikn/capstonedataset/paddleocr_pretrained/PP-OCRv5_server_rec_pretrained.pdparams
✓ Bukan model distilasi, tidak perlu diubah.
✓ Memori berhasil dibersihkan (Garbage Collected).\n
Membangun ulang file YAML secara dinamis...
✓ Config Detection siap.
✓ Config Recognition siap.


In [11]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", 
     "-r", "/kaggle/working/PaddleOCR/requirements.txt",
     "--dry-run",
     "--no-index",
     "--find-links", "/kaggle/input/datasets/herdinthorikn/capstonedataset/paddle_wheels/"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
print(result.stderr[-3000:])

kaggle/working/PaddleOCR/requirements.txt (line 2)) (1.16.3)




In [12]:
import os

# Cek 3 gambar pertama dari det_train.txt
with open("/kaggle/working/paddle_data/det_train_fixed.txt") as f:
    for i, line in enumerate(f):
        img_path = line.split("\t")[0]
        exists = os.path.exists(img_path)
        print(f"{'✓' if exists else '✗'} {img_path}")
        if i >= 2:
            break

✓ /kaggle/input/datasets/trainingdatapro/ocr-receipts-text-detection/images/1.jpg
✓ /kaggle/input/datasets/trainingdatapro/ocr-receipts-text-detection/images/16.jpg
✓ /kaggle/input/datasets/trainingdatapro/ocr-receipts-text-detection/images/13.jpg


In [13]:
import os
configs = [
    "/kaggle/working/PaddleOCR/configs/det/PP-OCRv5/PP-OCRv5_mobile_det.yml",
    "/kaggle/working/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml",
]
for c in configs:
    print(f"{'✓' if os.path.exists(c) else '✗'} {c}")

✓ /kaggle/working/PaddleOCR/configs/det/PP-OCRv5/PP-OCRv5_mobile_det.yml
✓ /kaggle/working/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml


In [14]:
import paddle
print("GPU available:", paddle.is_compiled_with_cuda())
print("GPU count:", paddle.device.cuda.device_count())
print("Current device:", paddle.device.get_device())

GPU available: True
GPU count: 2
Current device: gpu:0


In [15]:
# ============================================================
#  7 — Training Recognition
# ============================================================
import subprocess, sys, os

log_path = "/kaggle/working/logs/rec_train.log"
print(f"Log akan ditampilkan di layar HINGGA TUNTAS dan disimpan di → {log_path}\n")
print("-" * 60)

cmd = [
    sys.executable,
    "-m", "paddle.distributed.launch",
    "--gpus", "0,1",                        
    "/kaggle/working/PaddleOCR/tools/train.py",
    "-c", "/kaggle/working/configs/rec_finetune.yml"
]

with open(log_path, "w") as log_file:
    # Menggunakan Popen untuk membuka pipa (PIPE) komunikasi real-time
    process = subprocess.Popen(
        cmd,
        cwd="/kaggle/working/PaddleOCR",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,    # Mengubah output byte menjadi string
        bufsize=1     # Line-buffered (langsung cetak per baris)
    )

    # Menyedot output baris demi baris secara real-time
    for line in process.stdout:
        sys.stdout.write(line)  # 1. Cetak ke layar / console Kaggle
        log_file.write(line)    # 2. Tulis ke file rec_train.log
        log_file.flush()        # 3. Paksa simpan ke disk secara instan

    # Menahan cell agar tidak selesai sebelum GPU benar-benar tuntas
    process.wait()

print("-" * 60)
if process.returncode != 0:
    print(f"✗ Recognition training FAILED (code {process.returncode})")
else:
    print("✓ Detection training selesai")

Log akan ditampilkan di layar HINGGA TUNTAS dan disimpan di → /kaggle/working/logs/rec_train.log

------------------------------------------------------------
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
LAUNCH INFO 2026-06-04 00:43:50,020 -----------  Configuration  ----------------------
LAUNCH INFO 2026-06-04 00:43:50,020 auto_cluster_config: 0
LAUNCH INFO 2026-06-04 00:43:50,021 auto_parallel_config: None
LAUNCH INFO 2026-06-04 00:43:50,021 auto_tuner_json: None
LAUNCH INFO 2026-06-04 00:43:50,021 devices: 0,1
LAUNCH INFO 2026-06-04 00:43:50,021 elastic_level: -1
LAUNCH INFO 2026-06-04 00:43:50,021 elastic_timeout: 30
LAUNCH INFO 2026-06-04 00:43:50,021 enable_gpu_log: True
LAUNCH INFO 2026-06-04 00:43:50,021

In [16]:
# with open("/kaggle/working/logs/rec_train.log") as f:
#     lines = f.readlines()

# # Cari baris error pertama sebelum RecursionError
# for i, line in enumerate(lines):
#     if "error" in line.lower() or "Error" in line or "warning" in line.lower():
#         print(f"[{i}] {line}", end="")
#     if i > 200:
#         break

In [17]:
# ============================================================
#  7.1 - EXPORT, ONNX, & ZIP RECOGNITION
# ============================================================
import subprocess, sys, os, shutil

# 1. Setup Direktori
os.makedirs("/kaggle/working/output/rec_infer", exist_ok=True)
os.makedirs("/kaggle/working/onnx_models", exist_ok=True)

print("1. Mengekspor Model Rekognisi ke format Inference Paddle...")
res_export = subprocess.run([
    sys.executable, "/kaggle/working/PaddleOCR/tools/export_model.py",
    "-c", "/kaggle/working/configs/rec_finetune.yml",
    "-o", "Global.pretrained_model=/kaggle/working/output/rec_finetune/best_accuracy",
    "Global.save_inference_dir=/kaggle/working/output/rec_infer"
], cwd="/kaggle/working/PaddleOCR", capture_output=True, text=True)

if res_export.returncode != 0:
    print("✗ Export Rekognisi GAGAL:\n", res_export.stderr[-1000:])
else:
    print("✓ Export Inference Rekognisi sukses.")

    print("2. Mengkonversi Model Rekognisi ke format ONNX...")
    res_onnx = subprocess.run([
        "paddle2onnx",
        "--model_dir", "/kaggle/working/output/rec_infer",
        "--model_filename", "inference.pdmodel",
        "--params_filename", "inference.pdiparams",
        "--save_file", "/kaggle/working/onnx_models/hematin_recognition.onnx",
        "--input_shape_dict", "{'x': [-1, 3, 48, 320]}",
        "--opset_version", "11",
        "--enable_onnx_checker", "True"
    ], capture_output=True, text=True)
    
    if res_onnx.returncode != 0:
        print("✗ Konversi ONNX GAGAL:\n", res_onnx.stderr[-1000:])
    else:
        print("✓ Konversi ONNX Rekognisi sukses.")
        
        # 3. Salin Char Dict
        shutil.copy("/kaggle/working/configs/rec_char_dict.txt", "/kaggle/working/onnx_models/rec_char_dict.txt")
        print("✓ rec_char_dict.txt disalin.")
        
        # 4. ZIP
        print("3. Mengemas (Zipping) Asset Rekognisi untuk pengamanan...")
        shutil.make_archive("/kaggle/working/hematin_rec_secured", "zip", "/kaggle/working/onnx_models")
        size_mb = os.path.getsize("/kaggle/working/hematin_rec_secured.zip") / (1024*1024)
        print(f"★ BERHASIL! File hematin_rec_secured.zip ({size_mb:.1f} MB) sudah tersimpan aman di disk Kaggle.")

1. Mengekspor Model Rekognisi ke format Inference Paddle...
✓ Export Inference Rekognisi sukses.
2. Mengkonversi Model Rekognisi ke format ONNX...
✗ Konversi ONNX GAGAL:
 /usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
usage: paddle2onnx [-h] [--model_dir MODEL_DIR]
                   [--model_filename MODEL_FILENAME]
                   [--params_filename PARAMS_FILENAME] [--save_file SAVE_FILE]
                   [--opset_version OPSET_VERSION]
                   [--enable_auto_update_opset ENABLE_AUTO_UPDATE_OPSET]
                   [--enable_onnx_checker ENABLE_ONNX_CHECKER]
                   [--enable_dist_prim_all ENABLE_DIST_PRIM_ALL]
                   [--optimize_tool {polygraphy,onnxoptimizer,None}]
     

In [18]:
# ============================================================
#  8 — Training Detection
# ============================================================
import subprocess, sys, os

log_path = "/kaggle/working/logs/det_train.log"
print(f"Training detector (Single GPU)...")
print(f"Log akan ditampilkan di layar dan disimpan di → {log_path}\n")
print("-" * 60)

cmd = [
    sys.executable,
    "-m", "paddle.distributed.launch",
    "--gpus", "0",
    "/kaggle/working/PaddleOCR/tools/train.py",
    "-c", "/kaggle/working/configs/det_finetune.yml"
]

with open(log_path, "w") as log_file:
    # Menggunakan Popen untuk membuka pipa (PIPE) komunikasi real-time
    process = subprocess.Popen(
        cmd,
        cwd="/kaggle/working/PaddleOCR",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,    # Mengubah output byte menjadi string secara otomatis
        bufsize=1     # Line-buffered (langsung cetak per baris)
    )

    # Membaca output baris demi baris secara real-time
    for line in process.stdout:
        sys.stdout.write(line)  # 1. Cetak ke layar / console Kaggle
        log_file.write(line)    # 2. Tulis ke file det_train.log
        log_file.flush()        # 3. Paksa simpan ke disk saat itu juga

    # Tunggu hingga proses benar-benar selesai untuk mendapatkan return code
    process.wait()

print("-" * 60)
if process.returncode != 0:
    print(f"✗ Detection training FAILED (code {process.returncode})")
else:
    print("✓ Detection training selesai")

Training detector (Single GPU)...
Log akan ditampilkan di layar dan disimpan di → /kaggle/working/logs/det_train.log

------------------------------------------------------------
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
LAUNCH INFO 2026-06-04 02:06:20,946 -----------  Configuration  ----------------------
LAUNCH INFO 2026-06-04 02:06:20,946 auto_cluster_config: 0
LAUNCH INFO 2026-06-04 02:06:20,946 auto_parallel_config: None
LAUNCH INFO 2026-06-04 02:06:20,946 auto_tuner_json: None
LAUNCH INFO 2026-06-04 02:06:20,946 devices: 0
LAUNCH INFO 2026-06-04 02:06:20,947 elastic_level: -1
LAUNCH INFO 2026-06-04 02:06:20,947 elastic_timeout: 30
LAUNCH INFO 2026-06-04 02:06:20,947 enable_gpu_log: True
LAUNCH INFO 2026-

In [19]:
# with open("/kaggle/working/logs/det_train.log") as f:
#     lines = f.readlines()

# # Cari baris error pertama sebelum RecursionError
# for i, line in enumerate(lines):
#     if "error" in line.lower() or "Error" in line or "warning" in line.lower():
#         print(f"[{i}] {line}", end="")
#     if i > 200:
#         break

In [20]:
# ============================================================
#  8.2 - EXPORT, ONNX, & ZIP DETECTION
# ============================================================
import subprocess, sys, os, shutil

os.makedirs("/kaggle/working/output/det_infer", exist_ok=True)

print("➤ 1. Mengekspor Model Deteksi ke format Inference Paddle...")
res_export = subprocess.run([
    sys.executable, "/kaggle/working/PaddleOCR/tools/export_model.py",
    "-c", "/kaggle/working/configs/det_finetune.yml",
    "-o", "Global.pretrained_model=/kaggle/working/output/det_finetune/best_accuracy",
    "Global.save_inference_dir=/kaggle/working/output/det_infer"
], cwd="/kaggle/working/PaddleOCR", capture_output=True, text=True)

if res_export.returncode != 0:
    print("✗ Export Deteksi GAGAL:\n", res_export.stderr[-1000:])
else:
    print("✓ Export Inference Deteksi sukses.")

    print("2. Mengkonversi Model Deteksi ke format ONNX...")
    res_onnx = subprocess.run([
        "paddle2onnx",
        "--model_dir", "/kaggle/working/output/det_infer",
        "--model_filename", "inference.pdmodel",
        "--params_filename", "inference.pdiparams",
        "--save_file", "/kaggle/working/onnx_models/hematin_detector.onnx",
        "--input_shape_dict", "{'x': [-1, 3, -1, -1]}",
        "--opset_version", "11",
        "--enable_onnx_checker", "True"
    ], capture_output=True, text=True)
    
    if res_onnx.returncode != 0:
        print("✗ Konversi ONNX Deteksi GAGAL:\n", res_onnx.stderr[-1000:])
    else:
        print("✓ Konversi ONNX Deteksi sukses.")
        
        # 3. ZIP Final (Mencakup Rec dan Deteksi)
        print("3. Mengemas (Zipping) Seluruh Asset Final...")
        shutil.make_archive("/kaggle/working/hematin_models_complete", "zip", "/kaggle/working/onnx_models")
        size_mb = os.path.getsize("/kaggle/working/hematin_models_complete.zip") / (1024*1024)
        print(f"File hematin_models_complete.zip ({size_mb:.1f} MB)")

➤ 1. Mengekspor Model Deteksi ke format Inference Paddle...
✓ Export Inference Deteksi sukses.
2. Mengkonversi Model Deteksi ke format ONNX...
✗ Konversi ONNX Deteksi GAGAL:
 /usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
usage: paddle2onnx [-h] [--model_dir MODEL_DIR]
                   [--model_filename MODEL_FILENAME]
                   [--params_filename PARAMS_FILENAME] [--save_file SAVE_FILE]
                   [--opset_version OPSET_VERSION]
                   [--enable_auto_update_opset ENABLE_AUTO_UPDATE_OPSET]
                   [--enable_onnx_checker ENABLE_ONNX_CHECKER]
                   [--enable_dist_prim_all ENABLE_DIST_PRIM_ALL]
                   [--optimize_tool {polygraphy,onnxoptimizer,None}]
 

In [21]:
# # ============================================================
# #  9 — Export ke Paddle inference model
# # ============================================================
# def export_inference(config_path, best_model_path, infer_dir, label):
#     result = subprocess.run(
#         [
#             sys.executable,
#             "/kaggle/working/PaddleOCR/tools/export_model.py",
#             "-c", config_path,
#             "-o",
#             f"Global.pretrained_model={best_model_path}",
#             f"Global.save_inference_dir={infer_dir}",
#         ],
#         cwd="/kaggle/working/PaddleOCR",
#         capture_output=True, text=True,
#     )
#     if result.returncode != 0:
#         print(f"✗ Export {label} FAILED")
#         print(result.stderr[-2000:])
#     else:
#         print(f"✓ {label} inference model exported → {infer_dir}")
 
# export_inference(
#     "/kaggle/working/configs/det_finetune.yml",
#     "/kaggle/working/output/det_finetune/best_accuracy",
#     "/kaggle/working/output/det_infer",
#     "Detection"
# )
 
# export_inference(
#     "/kaggle/working/configs/rec_finetune.yml",
#     "/kaggle/working/output/rec_finetune/best_accuracy",
#     "/kaggle/working/output/rec_infer",
#     "Recognition"
# )

In [22]:
# ============================================================
#  9.5 — Inferensi validasi pada gambar struk baru
# ============================================================
import os

# os.makedirs("/kaggle/input/datasets/herdinthorikn/capstonedataset/OCR_Inference/OCR_Inference", exist_ok=True)
output_dir = "/kaggle/working/ocr_results/"
os.makedirs(output_dir, exist_ok=True)

print("Memulai proses inferensi...")
result = subprocess.run(
    [
        sys.executable,
        "/kaggle/working/PaddleOCR/tools/infer/predict_system.py",
        "--det_model_dir=/kaggle/working/output/det_infer",
        "--rec_model_dir=/kaggle/working/output/rec_infer",
        "--image_dir=/kaggle/input/datasets/herdinthorikn/capstonedataset/OCR_Inference/OCR_Inference/",
        "--rec_char_dict_path=/kaggle/working/configs/rec_char_dict.txt",
        "--use_gpu=True",
        "--draw_img_save_dir=/kaggle/working/ocr_results/",
    ],
    cwd="/kaggle/working/PaddleOCR",
    capture_output=True, text=True,
)

if result.returncode != 0:
    print("✗ Inferensi FAILED")
    print(result.stderr[-2000:])
    print("Visualisasi dibatalkan karena proses inferensi gagal.")
else:
    print("✓ Inferensi selesai")
    result_imgs = sorted(os.listdir(output_dir))
    
    if len(result_imgs) == 0:
        print("⚠ Peringatan: Inferensi sukses, tapi folder hasil kosong. Cek path dataset inputmu.")
    else:
        print(f"\nHasil disimpan: {len(result_imgs)} file")
        for img_name in result_imgs:
            print(f"\n── {img_name}")
            display(Image(filename=os.path.join(output_dir, img_name)))

Memulai proses inferensi...
✓ Inferensi selesai

Hasil disimpan: 6 file

── WhatsApp Image 2026-05-28 at 22.06.17.jpeg


NameError: name 'Image' is not defined

In [ ]:
# # ============================================================
# #  10 — Konversi ke ONNX
# # ============================================================
# def to_onnx(infer_dir, output_path, input_shape, label):
#     result = subprocess.run(
#         [
#             "paddle2onnx",
#             "--model_dir",        infer_dir,
#             "--model_filename",   "inference.pdmodel",
#             "--params_filename",  "inference.pdiparams",
#             "--save_file",        output_path,
#             "--input_shape_dict", input_shape,
#             "--opset_version",    "11",
#             "--enable_onnx_checker", "True",
#         ],
#         capture_output=True, text=True,
#     )
#     if result.returncode != 0:
#         print(f"✗ ONNX export {label} FAILED")
#         print(result.stderr[-2000:])
#     else:
#         size_mb = os.path.getsize(output_path) / 1024 / 1024
#         print(f"✓ {label} → {output_path} ({size_mb:.1f} MB)")
 
# to_onnx(
#     "/kaggle/working/output/det_infer",
#     "/kaggle/working/onnx_models/hematin_detector.onnx",
#     "{'x': [-1, 3, -1, -1]}",
#     "Detection"
# )
 
# to_onnx(
#     "/kaggle/working/output/rec_infer",
#     "/kaggle/working/onnx_models/hematin_recognition.onnx",
#     "{'x': [-1, 3, 48, 320]}",
#     "Recognition"
# )
 
# # Salin char dict ke folder onnx
# import shutil
# shutil.copy(dict_path, "/kaggle/working/onnx_models/rec_char_dict.txt")
# print("rec_char_dict.txt disalin")

In [ ]:
# ============================================================
#  11 — Verifikasi ONNX
# ============================================================
import onnxruntime as ort
import numpy as np
 
for model_path, dummy_shape, label in [
    ("/kaggle/working/onnx_models/hematin_detector.onnx", (1, 3, 960, 960), "Detection"),
    ("/kaggle/working/onnx_models/hematin_recognition.onnx", (1, 3, 48, 320),  "Recognition"),
]:
    session  = ort.InferenceSession(model_path, providers=["CUDAExecutionProvider"])
    inp_name = session.get_inputs()[0].name
    dummy    = np.random.randn(*dummy_shape).astype(np.float32)
    out      = session.run(None, {inp_name: dummy})
    print(f"✓ {label} — input: {dummy_shape}, output: {out[0].shape}")

In [ ]:
# ============================================================
#  12 — Zip hasil untuk diupload sebagai Dataset
# ============================================================
import shutil, os
 
out_zip = "/kaggle/working/hematin_onnx_models"
shutil.make_archive(out_zip, "zip", "/kaggle/working/onnx_models")
 
size_mb = os.path.getsize(f"{out_zip}.zip") / 1024 / 1024
print(f" {out_zip}.zip ({size_mb:.1f} MB)")
print("\nLangkah selanjutnya:")
print("  1. Download hematin_onnx_models.zip")
print("  2. Upload sebagai Kaggle Dataset baru: 'hematin-onnx-models'")
print("  3. Attach ke notebook classifier, load dari /kaggle/input/hematin-onnx-models/")
 